In [1]:
import dotenv
import openai
import os
import re
import shutil
import json
from pathlib import Path
from glob import glob
from langchain.llms.openai import OpenAI
from langchain.llms import AzureOpenAI


dotenv.load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
chatgpt_model_name = os.getenv('OPENAI_MODEL')
_llm_openai = OpenAI()

azure_deployment_name = os.getenv('AZURE_DEPLOYMENT')
azure_model_name = os.getenv('AZURE_MODEL')
# openai.api_type = "azure"
# openai.api_key = os.getenv("AZURE_OPENAI_KEY")
# openai.api_base = os.getenv('AZURE_ENDPOINT')
# openai.api_version = "2023-03-15-preview"

_llm_azure = AzureOpenAI(
    openai_api_key=os.getenv("AZURE_OPENAI_KEY"),
    openai_endpoint=os.getenv('AZURE_ENDPOINT'),
    openai_api_version="2023-03-15-preview",
    openai_api_type="azure",
    engine=os.getenv("AZURE_DEPLOYMENT"),
    model_name=os.getenv('AZURE_MODEL')
)

/home/deodoro/.local/lib/python3.11/site-packages/langchain/llms/openai.py:179: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
/home/deodoro/.local/lib/python3.11/site-packages/langchain/llms/openai.py:748: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


In [3]:
_llm_azure("what is azure openai")

AuthenticationError: <empty message>

In [ ]:
# Split audio files
from pydub import AudioSegment

def clean_out_path(fpath):
    out_path = os.path.join(fpath, 'out')

    # Check if the directory exists
    if os.path.exists(out_path):
        # Cleanup: delete all files in the directory
        for filename in os.listdir(out_path):
            file_path = os.path.join(out_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print('Failed to delete %s. Reason: %s' % (file_path, e))
    else:
        # Directory does not exist, so create it
        os.makedirs(out_path)

def split(audio, filename):
    ten_minutes = 20 * 60 * 1000
    pos = 0
    idx = 1
    while pos < len(audio):
        print("Saving {0}".format(filename.format(idx)))
        chunk = audio[pos:min(pos + ten_minutes, len(audio) - 1)]
        chunk.export(filename.format(idx), format="mp3")
        idx += 1
        pos += ten_minutes

for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir)
    for fname in glob("{0}/*.mp3".format(fdir)):
        print("Splitting {0}".format(fname))
        audio = AudioSegment.from_mp3(fname)
        outname = Path(os.path.splitext(fname)[0]).stem + " part {0}.mp3"
        split(audio, os.path.join(outdir, "out", outname))

In [ ]:
# Transcribe
for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir, "out")
    n = 1
    while n > 0:
        files = glob("{0}/*.mp3".format(outdir))
        n = 0
        for fname in files:
            outname = re.sub(r"\.mp3$", ".srt", fname)
            if not os.path.exists(outname):
                with open(fname, "rb") as audio_file:
                    print("transcribing {0}".format(fname))
                    transcript = openai.Audio.transcribe("whisper-1", audio_file, response_format="srt", api_key=OPENAI_API_KEY)
                    with open(outname, "wt") as srt_file:
                        srt_file.write(transcript)
                n += 1

In [ ]:
#Splitting text files (questions)
def save_chunk(prefix, chunk, file_count):
    with open(f'{prefix}{file_count:03d}.txt', 'w') as f:
        f.write(''.join(chunk))

def process_file(filename):
    size = 0
    chunk = []
    file_count = 1
    regex = re.compile(r'^[0-9]+\.')

    with open(filename, 'rt') as f:
        for line in f:
            if regex.match(line):
                save_chunk(filename, chunk, file_count)
                chunk = []
                size = 0
                file_count += 1

            size += len(line)
            chunk.append(line)

    # Save the last chunk if it's not empty
    if chunk:
        save_chunk(filename, chunk, file_count)

for f in glob("text/*.txt"):
    process_file(f)

In [ ]:
# Text files to OpenAI to clean JSON database
def f(text):
    messages=[
        {"role": "system", "content": "You are an assistant.This text contains one or two exam questions. Parse and produce a JSON array containing questions. each item should contain fields {enunciate, answer, explanation} extracted from the text. Each item has a question prompt, an answer and an explanation. Answer and explanation may be the same. Rewrite the explanation the best that you can, provided the original explanation. Enunciate is the transcription closest to original text of question prompt, fixed syntax and meaning. If there are more than one question, repeat the enunciate in each item, otherwise it is null. Output must be a JSON string parseable by python. Rewrite the text to fix syntax without changing the meaning."},
        {"role": "user", "content": text}
    ]
    # TO CHECK: I may have confused engine and model up in the beginning
    return openai.ChatCompletion.create(
            engine=azure_engine_name,
            model=azure_model_name,
            messages=messages,
            api_type = "azure",
            api_key = os.getenv("AZURE_OPENAI_KEY"),
            api_base = os.getenv('AZURE_ENDPOINT'),
            api_version = "2023-03-15-preview"
            )["choices"][0]["message"]["content"]

def process_file():
    with open('text/trimmed.txt', 'r') as file, open('text/processed.txt', 'w') as processed, open('text/output.json', 'w') as json_output:
        content = file.read()
        chunks = content.split('\n\n---\n\n')
        clean = []
        json_array = []

        c = 0
        for chunk in chunks:
            if chunk:
                print(".", end="")
                processed.write(chunk + '\n\n---\n\n')
                try:
                    clean = clean + [f(re.sub(r'[^\x20-\x7E]', '', chunk))]
                    processed.write(clean[-1])
                    try:
                        json_array = json_array + json.loads(clean[-1].replace('\\n', '\n').replace('\\\\', '\\'))
                    except:
                        processed.write('\nERROR PARSING JSON\n\n***\n\n')
                    processed.write('\n\n***\n\n')
                except:
                    processed.write("ERROR\n\n***\n\n")
        json.dump(json_array, json_output, indent=4)
        return clean

clean = process_file()

In [ ]:
def get_embeddings(text):
   if text:
      # TO CHECK: is the paramter openai_api_key or api_key?
      response = openai.Embedding.create(
          model="text-embedding-ada-002",
          input = text.replace("\n"," "),
          openai_api_key = OPENAI_API_KEY
      )

      embedding = response['data'][0]['embedding']
      return embedding
   else:
      return None


In [ ]:
from langchain.document_loaders import TextLoader
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

def format_response(enunciate, answer, explanation):
  if answer and explanation:
    if answer.upper() == "TRUE":
      return enunciate + "\n\nTrue. " + explanation
    elif answer.upper() == "FALSE":
      return enunciate + "\n\nFalse. " + explanation
    else:
      return enunciate + "\n\n" + answer + ". " + explanation
  elif answer:
    return enunciate + "\n\n" + answer
  elif explanation:
    return enunciate + "\n\n" + explanation
  else:
    return None

data = json.load(open('text/output.json', 'r'))
docs = []
print("Loading documents")
i = 0
for item in data:
    i += 1
    if i > 343:
      print("{0}/{1}".format(i, len(data)), end="\r")
      u = { "enunciate": item["enunciate"], \
            "answer": item["answer"], \
            "explanation": item["explanation"], \
            "combined" : format_response(item["enunciate"], item["answer"], item["explanation"]),
            "embedding_enunciate": get_embeddings(item["enunciate"]),\
            "embedding_answer": get_embeddings(item["answer"]),\
            "embedding_explanation": get_embeddings(item["explanation"]),\
            "embedding_combined": get_embeddings(format_response(item["enunciate"], item["answer"], item["explanation"]))}
      docs.append(u)
print("Done loading documents")
with open('json_embedded.json', 'wt') as f:
    f.write(json.dumps(docs, indent=2))

In [ ]:
import psycopg
from psycopg.conninfo import make_conninfo

pg_string = make_conninfo(
  host=os.getenv('DB_HOST'),
  port=os.getenv('DB_PORT'),
  dbname=os.getenv('DB_DATABASE'),
  user=os.getenv('DB_USER'),
  password=os.getenv('DB_PASSWORD')
)


table_sql = """
drop table docs;
create table docs (
  enunciate text not null,
  answer text,
  explanation text,
  combined text not null,
  embedding_enunciate vector(1536),
  embedding_answer vector(1536),
  embedding_explanation vector(1536),
  embedding_combined vector(1536)
);
ALTER TABLE docs ADD CONSTRAINT unique_enunciate UNIQUE (enunciate);
"""

with psycopg.connect(pg_string) as conn:
    conn.execute(table_sql)


In [ ]:
with psycopg.connect(pg_string) as conn:
    docs = json.loads(open('json_embedded.json', 'r').read())
    sql = "INSERT INTO docs (enunciate, answer, explanation, combined, embedding_enunciate, embedding_answer, embedding_explanation, embedding_combined) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (enunciate) DO NOTHING;"

    for doc in docs:
        if doc["combined"]:
            conn.execute(sql, (doc["enunciate"], doc["answer"], doc["explanation"], doc["combined"], doc["embedding_enunciate"], doc["embedding_answer"], doc["embedding_explanation"], doc["embedding_combined"]))
        else:
            print(doc)
        conn.commit()

In [ ]:
query = get_embeddings("Cornout collusion")

with psycopg.connect(pg_string) as conn:
    results = conn.execute("SELECT * FROM docs ORDER BY embedding_combined <=> %s::vector LIMIT 5;", [query])
    out = results.fetchall()

titles = [row[0] for row in out]
print("\n".join(titles))

In [11]:
from langchain.document_loaders import DirectoryLoader
from langchain.embeddings.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
loader_lectures = DirectoryLoader("./lectures/")
lectures = loader_lectures.load()
loader_books = DirectoryLoader("./books/")
books = loader_books.load()

In [ ]:
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

conn_string = "postgresql://{0}:{1}@{2}:{3}/{4}".format(os.getenv('DB_USER'), os.getenv('DB_PASSWORD'), os.getenv('DB_HOST'), os.getenv('DB_PORT'), os.getenv('DB_DATABASE'))
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator=" ")

lectures_docs = text_splitter.split_documents(lectures)
db_lectures = PGVector.from_documents(embedding=embeddings, documents=lectures_docs, collection_name="lectures", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

books_docs = text_splitter.split_documents(books)
db_books = PGVector.from_documents(embedding=embeddings, documents=books_docs, collection_name="books", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

In [ ]:
[c.page_content for c,r in db_lectures.similarity_search_with_score("Cornout collusion", 10)]
[c.page_content for c,r in db_books.similarity_search_with_score("Cornout collusion", 10)]

In [ ]:
import psycopg
from psycopg.conninfo import make_conninfo
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

pg_string = make_conninfo(
  host=os.getenv('DB_HOST'),
  port=os.getenv('DB_PORT'),
  dbname=os.getenv('DB_DATABASE'),
  user=os.getenv('DB_USER'),
  password=os.getenv('DB_PASSWORD')
)

conn_string = "postgresql://{0}:{1}@{2}:{3}/{4}".format(os.getenv('DB_USER'), os.getenv('DB_PASSWORD'), os.getenv('DB_HOST'), os.getenv('DB_PORT'), os.getenv('DB_DATABASE'))

# with psycopg.connect(pg_string) as conn:
#     docs = json.loads(open('json_embedded.json', 'r').read())
#     sql = "INSERT INTO docs (enunciate, answer, explanation, combined, embedding_enunciate, embedding_answer, embedding_explanation, embedding_combined) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (enunciate) DO NOTHING;"

#     for doc in docs:
#         if doc["combined"]:
#             conn.execute(sql, (doc["enunciate"], doc["answer"], doc["explanation"], doc["combined"], doc["embedding_enunciate"], doc["embedding_answer"], doc["embedding_explanation"], doc["embedding_combined"]))
#         else:
#             print(doc)
#         conn.commit()

db_lectures = PGVector(embedding_function=embeddings,collection_name="lectures", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)
db_books = PGVector(embedding_function=embeddings, collection_name="books", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

In [ ]:
with psycopg.connect(pg_string) as conn:
    docs = json.loads(open('json_embedded.json', 'r').read())
    sql = "INSERT INTO docs (enunciate, answer, explanation, combined, embedding_enunciate, embedding_answer, embedding_explanation, embedding_combined) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (enunciate) DO NOTHING;"

    for doc in docs:
        if doc["combined"]:
            conn.execute(sql, (doc["enunciate"], doc["answer"], doc["explanation"], doc["combined"], doc["embedding_enunciate"], doc["embedding_answer"], doc["embedding_explanation"], doc["embedding_combined"]))
        else:
            print(doc)
        conn.commit()

In [ ]:
from langchain.chains import RetrievalQA

retriever = db_books.as_retriever(search_type="similarity", search_kwargs={"k":2})
qa = RetrievalQA.from_chain_type(llm=_llm_azure, chain_type="stuff", retriever=retriever, return_source_documents=True)
query = "Cornout collusion?"
result = qa({"query": query})
print(result)

In [ ]:
u = db_books.similarity_search("Cornout collusion", 10)
print(u)

In [ ]:
db_books